In [1]:
pwd

'/Users/bethany/Computational Modeling and Simulations/modeling condensate project/An-Integer-Programming-Framework-for-Inferring-Biomolecular-Condensate-Formation-/data_cleaning'

In [2]:
import os
import pandas as pd

In [3]:
!pip install openpyxl xlrd

In [4]:
import os
import pandas as pd

# 1. Define file paths
base_dir = "/Users/bethany/Computational Modeling and Simulations/modeling condensate project"
pos_path = os.path.join(base_dir, "phase_separation_Unambiguous", "LLPS.xls")
neg_path = os.path.join(base_dir, "No_phase_separation_Unambiguous", "LLPS.xls")

# 2. load excel files w multi lines
def load_llps_file(filepath):
    try:
        return pd.read_excel(filepath)
    except Exception:
        return pd.read_csv(filepath, sep="\t", engine="python", encoding="utf-8", on_bad_lines="skip")

# 3. Read both files into separate DataFrames
df_pos = load_llps_file(pos_path)
df_neg = load_llps_file(neg_path)

df_pos.columns = df_pos.columns.str.strip()
df_neg.columns = df_neg.columns.str.strip()

# 4. Print shapes and compare column names to make sure they are the same before filtering
print(f"Positive shape: {df_pos.shape}")
print(f"Negative shape: {df_neg.shape}\n")

print("POSITIVE COLUMNS")
print(list(df_pos.columns), "\n")

print("NEGATIVE COLUMNS")
print(list(df_neg.columns), "\n")

print("Only in Positive:", set(df_pos.columns) - set(df_neg.columns))
print("Only in Negative:", set(df_neg.columns) - set(df_pos.columns))


Positive shape: (4111, 31)
Negative shape: (1894, 31)

POSITIVE COLUMNS
['PSID', 'Entry ID', 'Components type', 'Protein ID', 'Protein name', 'Protein type (N/D)', 'Fusion', 'Cleaved', 'Repeat', 'Mutation', 'PTM', 'Sequence length', 'Sequence', 'Protein structure type', 'Nucleic acid', 'Solute concentration', 'Salt concentration', 'Buffer', 'Crowding agent', 'Temperature', 'Pressure', 'Incubation time', 'other molecules', 'Detection method', 'Phase separation', 'Phase diagram', 'Morphology', 'Description', 'PMID', 'DOI', 'In vivo/In cell'] 

NEGATIVE COLUMNS
['PSID', 'Entry ID', 'Components type', 'Protein ID', 'Protein name', 'Protein type (N/D)', 'Fusion', 'Cleaved', 'Repeat', 'Mutation', 'PTM', 'Sequence length', 'Sequence', 'Protein structure type', 'Nucleic acid', 'Solute concentration', 'Salt concentration', 'Buffer', 'Crowding agent', 'Temperature', 'Pressure', 'Incubation time', 'other molecules', 'Detection method', 'Phase separation', 'Phase diagram', 'Morphology', 'Descripti

In [5]:
#Inspect unique values in 'Components type' first so nothing gets missed
print("Positive Components types:\n", df_pos["Components type"].value_counts(), "\n")
print("Negative Components types:\n", df_neg["Components type"].value_counts(), "\n")

Positive Components types:
 Components type
protein(1)           2020
protein(1) + RNA      643
protein(2)            519
protein(1) + DNA      273
Protein(2)            183
protein(3)             79
protein(2) + RNA       77
Protein(1)             72
Protein(3)             41
protein(4)             37
Protein(1) + RNA       35
protein(2) + DNA       19
protein(6)             15
protein(5)             15
Protein(2) + RNA       15
protein(1)              9
Protein(1) + DNA        9
protein(1) +  RNA       8
Protein(3) + RNA        7
Protein(3) + DNA        7
protein(2) +  RNA       5
protein(7)              4
protein(3) + RNA        4
Protein(2)              4
Protein(2) + DNA        4
Protein(4)              2
protein(2)              2
protein(3) + DNA        2
protein(4) + DNA        1
Name: count, dtype: int64 

Negative Components types:
 Components type
protein(1)           1110
protein(1) + RNA      307
protein(2)            182
protein(1) + DNA       96
Protein(2)             61


In [6]:
#they are inconsistent with labeling and capitalization so we have to take this into account for the filtering

In [7]:
#normalizing the text and spacing inside the columns
import re

def normalize_columns(df):
    d = df.copy()
    
    # strip leading/trailing spaces from column headers
    d.columns = d.columns.str.strip()
    
    # Clean 'Components type' column: trim edges, lowercase, and collapse double spaces
    d["Components type"] = (
        d["Components type"]
        .astype(str)
        .str.strip()
        .str.lower()
        .apply(lambda x: re.sub(r"\s+", " ", x))
    )
    
    # Clean 'Protein type (N/D)': trim edges and uppercase ('n' -> 'N', 'd' -> 'D')
    d["Protein type (N/D)"] = d["Protein type (N/D)"].astype(str).str.strip().str.upper()
    
    # Clean wild-type check columns: trim edges so ' - ' becomes '-'
    wt_cols = ["Fusion", "Cleaved", "Repeat", "Mutation", "PTM"]
    for col in wt_cols:
        d[col] = d[col].astype(str).str.strip()
        
    # Clean 'Nucleic acid' and 'Protein ID' whitespace
    d["Nucleic acid"] = d["Nucleic acid"].astype(str).str.strip()
    d["Protein ID"] = d["Protein ID"].astype(str).str.strip()
    
    return d

# Normalize both DataFrames 
df_pos = normalize_columns(df_pos)
df_neg = normalize_columns(df_neg)

# check the cleaned 'Components type' counts to make sure theres only 1 unique way for each column now
print("CLEANED POS COMPONENTS")
print(df_pos["Components type"].value_counts(), "\n")

print("CLEANED NEG COMPONENTS")
print(df_neg["Components type"].value_counts())


CLEANED POS COMPONENTS
Components type
protein(1)          2101
protein(2)           708
protein(1) + rna     686
protein(1) + dna     282
protein(3)           120
protein(2) + rna      97
protein(4)            39
protein(2) + dna      23
protein(6)            15
protein(5)            15
protein(3) + rna      11
protein(3) + dna       9
protein(7)             4
protein(4) + dna       1
Name: count, dtype: int64 

CLEANED NEG COMPONENTS
Components type
protein(1)          1165
protein(1) + rna     324
protein(2)           247
protein(1) + dna      99
protein(3)            27
protein(2) + rna      13
protein(4)             9
protein(2) + dna       3
protein(3) + dna       3
protein(5)             3
protein(3) + rna       1
Name: count, dtype: int64


In [8]:
# Check unique values in Protein type (N/D) for both files
print("POS Protein type (N/D)")
print(df_pos["Protein type (N/D)"].value_counts(), "\n")

print("NEG Protein type (N/D)")
print(df_neg["Protein type (N/D)"].value_counts(), "\n")


POS Protein type (N/D)
Protein type (N/D)
N                2636
N;N               731
D                 435
N;N;N             126
N;D                56
N;N;N;N            37
D;D                26
N;N;N;N;N;D        15
N;N;N;N;N          11
N;N;D              10
D;N                 5
N;N;D;N;N           4
N;N;N;N;N;D;N       4
N;N;D;N             3
N;D;D               3
D,N;N               2
D,N                 2
D,N;N,D;N,D         1
D,N;N,D             1
D,N,D               1
N,D,N               1
N,D,N;N,D,N         1
Name: count, dtype: int64 

NEG Protein type (N/D)
Protein type (N/D)
N            1450
N;N           211
D             138
N;D            38
N;N;N          29
N;N;N;N         9
D;D             9
N;N;N;N;N       3
D,N;N           2
N,D             1
N,D,N           1
N;N;D           1
N;D;D           1
D,N             1
Name: count, dtype: int64 



In [9]:
import re

#now that I saw what the N/D columns look like i want to filter by that column one time so that i am checking and then makign 
#decisions step by step together. i want to now first find a total of number of columns in each data set that have D listed anywhere 
#in that combination compared to the total number of cells and then i can use that and subtract the number of columns with designed synthetic 
# proteins so i can check the amount of data i would be potentially throwing away

def is_pure_natural(val):
    # Split by semicolon or comma and check that every protein in the row is 'N'
    tokens = [t.strip().upper() for t in re.split(r"[;,]", str(val)) if t.strip()]
    return len(tokens) > 0 and all(t == "N" for t in tokens)

def audit_protein_type(df, name):
    total = len(df)
    pure_n_mask = df["Protein type (N/D)"].apply(is_pure_natural)
    has_d_mask = ~pure_n_mask
    
    print(f"{name}")
    print(f"Total rows:               {total}")
    print(f"Rows with 'D' anywhere:   {has_d_mask.sum()}  (would be dropped)")
    print(f"Remaining pure 'N' rows:  {pure_n_mask.sum()}  ({total} - {has_d_mask.sum()})\n")
    return pure_n_mask

pos_n_mask = audit_protein_type(df_pos, "POSITIVE DATASET")
neg_n_mask = audit_protein_type(df_neg, "NEGATIVE DATASET")

POSITIVE DATASET
Total rows:               4111
Rows with 'D' anywhere:   570  (would be dropped)
Remaining pure 'N' rows:  3541  (4111 - 570)

NEGATIVE DATASET
Total rows:               1894
Rows with 'D' anywhere:   192  (would be dropped)
Remaining pure 'N' rows:  1702  (1894 - 192)



In [10]:
#there is a good amount of data still left over if i remove experiments with synthetic designed molecules. We will drop these

# apply the pure 'N' filter to drop all rows containing 'D'
df_pos_n = df_pos[pos_n_mask].copy()
df_neg_n = df_neg[neg_n_mask].copy()

# verify shapes match the previous expected counts (3541 and 1702)
print("New Positive shape:", df_pos_n.shape)
print("New Negative shape:", df_neg_n.shape, "\n")

# verify that only pure 'N' patterns remain in both datasets by checking unique patterns/values again in those columns
print("VERIFIED POS Protein type (N/D) ---")
print(df_pos_n["Protein type (N/D)"].value_counts(), "\n")

print("VERIFIED NEG Protein type (N/D) ---")
print(df_neg_n["Protein type (N/D)"].value_counts())


New Positive shape: (3541, 31)
New Negative shape: (1702, 31) 

VERIFIED POS Protein type (N/D) ---
Protein type (N/D)
N            2636
N;N           731
N;N;N         126
N;N;N;N        37
N;N;N;N;N      11
Name: count, dtype: int64 

VERIFIED NEG Protein type (N/D) ---
Protein type (N/D)
N            1450
N;N           211
N;N;N          29
N;N;N;N         9
N;N;N;N;N       3
Name: count, dtype: int64


In [11]:
# the protein type filtering step was successful and preserved only experiments with natural proteins.

In [12]:
def audit_components(df, name):
    total = len(df)
    multi_mask = df["Components type"] != "protein(1)"
    single_mask = ~multi_mask
    
    print(f"{name}")
    print(f"Current rows:                  {total}")
    print(f"Single-protein ('protein(1)'): {single_mask.sum()}  (would be dropped)")
    print(f"Remaining multi-molecule rows: {multi_mask.sum()}  ({total} - {single_mask.sum()})\n")
    print("Breakdown of remaining multi-molecule rows:")
    print(df[multi_mask]["Components type"].value_counts(), "\n")
    return multi_mask

pos_multi_mask = audit_components(df_pos_n, "POSITIVE DATASET (df_pos_n)")
neg_multi_mask = audit_components(df_neg_n, "NEGATIVE DATASET (df_neg_n)")


POSITIVE DATASET (df_pos_n)
Current rows:                  3541
Single-protein ('protein(1)'): 1798  (would be dropped)
Remaining multi-molecule rows: 1743  (3541 - 1798)

Breakdown of remaining multi-molecule rows:
Components type
protein(2)          639
protein(1) + rna    575
protein(1) + dna    257
protein(3)          106
protein(2) + rna     76
protein(4)           36
protein(2) + dna     22
protein(3) + rna     11
protein(5)           11
protein(3) + dna      9
protein(4) + dna      1
Name: count, dtype: int64 

NEGATIVE DATASET (df_neg_n)
Current rows:                  1702
Single-protein ('protein(1)'): 1071  (would be dropped)
Remaining multi-molecule rows: 631  (1702 - 1071)

Breakdown of remaining multi-molecule rows:
Components type
protein(1) + rna    280
protein(2)          198
protein(1) + dna     96
protein(3)           25
protein(2) + rna     13
protein(4)            9
protein(2) + dna      3
protein(3) + dna      3
protein(5)            3
protein(3) + rna      1
Name:

In [13]:
#remaining multimolecule experiments would be 1743 (pos-forming condensates) + 631 (not forming condensates) = 2374 total  
# which is still a good size dataset. so i will drop any experiments that only had one protein molecule 

# apply the multi-molecule filter
df_pos_multi = df_pos_n[pos_multi_mask].copy()
df_neg_multi = df_neg_n[neg_multi_mask].copy()

# check shapes match the expected counts (1743 and 631)
print("New Positive shape:", df_pos_multi.shape)
print("New Negative shape:", df_neg_multi.shape, "\n")

# double check that 'protein(1)' is completely gone from both datasets
print("Any 'protein(1)' left in Positive?", (df_pos_multi["Components type"] == "protein(1)").any())
print("Any 'protein(1)' left in Negative?", (df_neg_multi["Components type"] == "protein(1)").any())


New Positive shape: (1743, 31)
New Negative shape: (631, 31) 

Any 'protein(1)' left in Positive? False
Any 'protein(1)' left in Negative? False


In [14]:
wt_cols = ["Fusion", "Cleaved", "Repeat", "Mutation", "PTM"]

def is_all_wildtype_cell(val):
    s = str(val).strip()
    if s in ["", "-", "nan", "None"]:
        return True
    tokens = [t.strip() for t in re.split(r"[;,]", s) if t.strip()]
    return len(tokens) > 0 and all(t == "-" for t in tokens)

def audit_wildtype(df, name):
    total = len(df)
    print(f"{name} (Current rows: {total})")
    
    combined_mask = pd.Series(True, index=df.index)
    for col in wt_cols:
        col_mask = df[col].apply(is_all_wildtype_cell)
        dropped = (~col_mask).sum()
        print(f"  {col:<10} -> modified rows: {dropped}")
        combined_mask = combined_mask & col_mask
        
    print(f"\nStrict Wild-Type remaining: {combined_mask.sum()} / {total}\n")
    return combined_mask

pos_wt_mask = audit_wildtype(df_pos_multi, "POSITIVE DATASET (df_pos_multi)")
neg_wt_mask = audit_wildtype(df_neg_multi, "NEGATIVE DATASET (df_neg_multi)")


POSITIVE DATASET (df_pos_multi) (Current rows: 1743)
  Fusion     -> modified rows: 96
  Cleaved    -> modified rows: 980
  Repeat     -> modified rows: 66
  Mutation   -> modified rows: 261
  PTM        -> modified rows: 113

Strict Wild-Type remaining: 627 / 1743

NEGATIVE DATASET (df_neg_multi) (Current rows: 631)
  Fusion     -> modified rows: 17
  Cleaved    -> modified rows: 342
  Repeat     -> modified rows: 17
  Mutation   -> modified rows: 94
  PTM        -> modified rows: 53

Strict Wild-Type remaining: 237 / 631



In [15]:
#consider dropping out only Mutation and Repeat columns since those correspond to mutating the molecules, having tandem repeats, 

print("Positive keeping Fusion & PTM (dropping Repeat/Mutation):", (df_pos_multi["Repeat"].apply(is_all_wildtype_cell) & df_pos_multi["Mutation"].apply(is_all_wildtype_cell)).sum())
print("Negative keeping Fusion & PTM (dropping Repeat/Mutation):", (df_neg_multi["Repeat"].apply(is_all_wildtype_cell) & df_neg_multi["Mutation"].apply(is_all_wildtype_cell)).sum())


Positive keeping Fusion & PTM (dropping Repeat/Mutation): 1431
Negative keeping Fusion & PTM (dropping Repeat/Mutation): 524


In [16]:
#so there would still be 681 pos and 257 neg experiments left

In [17]:
#apply filter for repeats and mutations to drop them
# keep cleaved fragments (same protein / same UniProt ID); drop mutated and tandem-repeat constructs
wt_mask_pos = df_pos_multi["Repeat"].apply(is_all_wildtype_cell) & df_pos_multi["Mutation"].apply(is_all_wildtype_cell)
wt_mask_neg = df_neg_multi["Repeat"].apply(is_all_wildtype_cell) & df_neg_multi["Mutation"].apply(is_all_wildtype_cell)

df_pos_wt = df_pos_multi[wt_mask_pos].copy()
df_neg_wt = df_neg_multi[wt_mask_neg].copy()

print("New Positive shape:", df_pos_wt.shape)   # expect (1431, 31)
print("New Negative shape:", df_neg_wt.shape)   # expect (524, 31)

New Positive shape: (1431, 31)
New Negative shape: (524, 31)


In [20]:
df_pos_wt.head()

,PSID,Entry ID,Components type,Protein ID,Protein name,Protein type (N/D),Fusion,Cleaved,Repeat,Mutation,...,Incubation time,other molecules,Detection method,Phase separation,Phase diagram,Morphology,Description,PMID,DOI,In vivo/In cell
1,PS00000011,LLPS000109,protein(1) + rna,p0003,Whi3,N,-,-,-,-,...,NaN,0.01-0.07 mg/ml dextran 10-155 kD,Microscopy,Yes,NaN,droplet,The smaller 10 kDa dextranpartitions favourabl...,29064502.0,NaN,Yes
2,PS00000012,LLPS001033,protein(1) + rna,p0225,xNPM1,N,-,-,-,-,...,NaN,0.01-0.07 mg/ml dextran 10-155 kD,Microscopy,Yes,NaN,droplet,The smaller 10 kDa dextranpartitions favourabl...,29064502.0,NaN,Yes
33,PS00000058,LLPS000146,protein(1) + rna,p0006,elF4GII_m,N,-,(IDR),-,-,...,NaN,-,Microscopy,Yes,NaN,droplet,A range of RNA concentrations (0.1–10 mM) did ...,26412307.0,NaN,NaN
36,PS00000064,LLPS000180,protein(2),p0008;p0006,hnRNPA1 + elF4GII_m,N;N,-;-,-;(IDR),-;-,-;-,...,after 45 min,-,Microscopy,Yes,NaN,droplet,"The Pub1 IDR, Fus IDR, eIF4GII IDR, and Lsm4 I...",26412307.0,NaN,NaN
37,PS00000065,LLPS000177,protein(2),p0008;p0002,hnRNPA1 + FUS_m,N;N,-;-,-;(IDR),-;-,-;-,...,after 45 min,-,Microscopy,Yes,NaN,droplet,"The Pub1 IDR, Fus IDR, eIF4GII IDR, and Lsm4 I...",26412307.0,NaN,NaN


In [22]:
pd.set_option("display.max_rows", None)
df_pos_wt["Phase diagram"].value_counts(dropna=False)   # dropna=False also counts empty cells

Phase diagram
NaN    1431
Name: count, dtype: int64

In [23]:
#drop description, DOI, in vivo, phase diagram, detection method, incubation time, 
drop_cols = ["Description", "DOI", "In vivo/In cell", "Phase diagram", "Detection method", "Incubation time"]

df_pos_wt = df_pos_wt.drop(columns=drop_cols)
df_neg_wt = df_neg_wt.drop(columns=drop_cols)

print("Positive shape:", df_pos_wt.shape)   # expect (1431, 25)
print("Negative shape:", df_neg_wt.shape)   # expect (524, 25)
print(list(df_pos_wt.columns))

Positive shape: (1431, 25)
Negative shape: (524, 25)
['PSID', 'Entry ID', 'Components type', 'Protein ID', 'Protein name', 'Protein type (N/D)', 'Fusion', 'Cleaved', 'Repeat', 'Mutation', 'PTM', 'Sequence length', 'Sequence', 'Protein structure type', 'Nucleic acid', 'Solute concentration', 'Salt concentration', 'Buffer', 'Crowding agent', 'Temperature', 'Pressure', 'other molecules', 'Phase separation', 'Morphology', 'PMID']


In [ ]:
#drop temperature, pressure, morphology, 

In [24]:
for col in ["Repeat", "Mutation"]:
    print(col, df_pos_wt[col].apply(is_all_wildtype_cell).all(), df_neg_wt[col].apply(is_all_wildtype_cell).all())

Repeat True True
Mutation True True


In [25]:
df_pos_wt = df_pos_wt.drop(columns=["Repeat", "Mutation", "Cleaved", "PTM"])
df_neg_wt = df_neg_wt.drop(columns=["Repeat", "Mutation", "Cleaved", "PTM"])

In [26]:
#drop temperature, pressure, morphology, 
df_pos_wt = df_pos_wt.drop(columns=["Temperature", "Pressure", "Morphology"])
df_neg_wt = df_neg_wt.drop(columns=["Temperature", "Pressure", "Morphology"])

In [27]:
df_pos_wt.head()

,PSID,Entry ID,Components type,Protein ID,Protein name,Protein type (N/D),Fusion,Sequence length,Sequence,Protein structure type,Nucleic acid,Solute concentration,Salt concentration,Buffer,Crowding agent,other molecules,Phase separation,PMID
1,PS00000011,LLPS000109,protein(1) + rna,p0003,Whi3,N,-,[Whi3]:729aa(1-729),>LLPS000109 tr|Q75E28|Q75E28_ASHGO ABL158Cp OS...,fold,BNI1:6590nt(1-6590),9 µM [Whi3];50 nM BNI1,150 mM KCl,"20 mM Tris, pH 7.4, 1 mM DTT",-,0.01-0.07 mg/ml dextran 10-155 kD,Yes,29064502.0
2,PS00000012,LLPS001033,protein(1) + rna,p0225,xNPM1,N,-,[xNPM1]:299aa(1-299),>LLPS001033 sp|P07222|NPM_XENLA Nucleophosmin ...,fold,RNA,25 µM [xNPM1];0.1 mg/ml rRNA,150 mM NaCl,"20 mM Tris, pH 7.4, 1 mM DTT",-,0.01-0.07 mg/ml dextran 10-155 kD,Yes,29064502.0
33,PS00000058,LLPS000146,protein(1) + rna,p0006,elF4GII_m,N,-,[elF4GII IDR]:85aa(13-97),>LLPS000146 sp|P39936|IF4F2_YEAST Eukaryotic i...,IDR-fold,RNA_(PTB ligand):50nt,30 µM [elF4GII IDR];0.1-4 µM RNA,100 mM NaCl,"20 mM Tris pH 7.4, 1 mM DTT",-,-,Yes,26412307.0
36,PS00000064,LLPS000180,protein(2),p0008;p0006,hnRNPA1 + elF4GII_m,N;N,-;-,[hnRNPA1]:320aa(1-320)|[elF4GII IDR]:85aa(13-97),>LLPS000180.1 sp|P09651-2|ROA1_HUMAN Isoform A...,IDR-fold + IDR-fold,-,30 µM [hnRNPA1];100 nM [elF4GII IDR],37.5 mM NaCl,20 mM Tris pH 7.4,-,-,Yes,26412307.0
37,PS00000065,LLPS000177,protein(2),p0008;p0002,hnRNPA1 + FUS_m,N;N,-;-,[hnRNPA1]:320aa(1-320)|[FUS IDR]:237aa(1-237),>LLPS000177.1 sp|P09651-2|ROA1_HUMAN Isoform A...,IDR-fold + IDR,-,30 µM [hnRNPA1];100 nM [FUS IDR],37.5 mM NaCl,20 mM Tris pH 7.4,-,-,Yes,26412307.0


In [28]:
# drop sequence, sequence length, protein type,protein structure type, nucleic acid, solute concentration, 
#salt concentation, buffer, crowding agent, other molecules, 
drop_now = ["Sequence", "Sequence length", "Protein type (N/D)", "Protein structure type",
            "Salt concentration", "Buffer", "Crowding agent", "other molecules", "Solute concentration"]

df_pos_wt = df_pos_wt.drop(columns=drop_now)
df_neg_wt = df_neg_wt.drop(columns=drop_now)
print(list(df_pos_wt.columns))

['PSID', 'Entry ID', 'Components type', 'Protein ID', 'Protein name', 'Fusion', 'Nucleic acid', 'Phase separation', 'PMID']


In [29]:
def audit_nucleic(df, name):
    na_mask = df["Components type"].str.contains("rna|dna")
    print(f"{name}")
    print(f"Current rows:            {len(df)}")
    print(f"Rows with RNA or DNA:    {na_mask.sum()}  (would be dropped)")
    print(f"Remaining protein-only:  {(~na_mask).sum()}\n")
    print(df.loc[na_mask, "Nucleic acid"].value_counts().head(10), "\n")
    return ~na_mask

pos_prot_mask = audit_nucleic(df_pos_wt, "POSITIVE DATASET")
neg_prot_mask = audit_nucleic(df_neg_wt, "NEGATIVE DATASET")

POSITIVE DATASET
Current rows:            1431
Rows with RNA or DNA:    855  (would be dropped)
Remaining protein-only:  576

Nucleic acid
60 bp duplex oligo(GAAGCTAGACTTAGGTGTCATATTGAACCTACTATGCCGAACTAGTTACGAGCTATAAAC)                                                            33
total RNAs                                                                                                                                  29
total RNA_(Total RNA was isolated from HeLa cells using the RNeasy Mini Kit (QIAGEN) and following the manufacturer’s instructions.)        27
H20_(20-nt ssRNA labeled with HEX,sequence:AGAUUCAACUGGCAGUAACC)                                                                            26
RNA_(PTB ligand):50nt                                                                                                                       24
NEAT1 RNA                                                                                                                                   21
dT7

In [30]:
# drop only "protein(1) + rna/dna" rows: once the RNA is ignored there is just one protein, so no pair.
# keep "protein(2+) + rna/dna" rows: the proteins still co-occurred; we just ignore the nucleic acid.
pos_single_na = df_pos_wt["Components type"].str.match(r"protein\(1\) \+ (rna|dna)")
neg_single_na = df_neg_wt["Components type"].str.match(r"protein\(1\) \+ (rna|dna)")

print("Positive protein(1)+RNA/DNA rows dropped:", pos_single_na.sum())   # expect 748
print("Negative protein(1)+RNA/DNA rows dropped:", neg_single_na.sum())   # expect 343

df_pos_p = df_pos_wt[~pos_single_na].copy()
df_neg_p = df_neg_wt[~neg_single_na].copy()

# record whether RNA/DNA was in the tube, before dropping the column
df_pos_p["with_nucleic_acid"] = df_pos_p["Components type"].str.contains("rna|dna")
df_neg_p["with_nucleic_acid"] = df_neg_p["Components type"].str.contains("rna|dna")

df_pos_p = df_pos_p.drop(columns=["Nucleic acid"])
df_neg_p = df_neg_p.drop(columns=["Nucleic acid"])

print("New Positive shape:", df_pos_p.shape)   # expect 683 rows
print("New Negative shape:", df_neg_p.shape)   # expect 181 rows
print(df_pos_p["Components type"].value_counts())

Positive protein(1)+RNA/DNA rows dropped: 748
Negative protein(1)+RNA/DNA rows dropped: 343
New Positive shape: (683, 9)
New Negative shape: (181, 9)
Components type
protein(2)          482
protein(3)           67
protein(2) + rna     64
protein(2) + dna     22
protein(4)           21
protein(3) + rna     11
protein(3) + dna      9
protein(5)            6
protein(4) + dna      1
Name: count, dtype: int64


In [31]:
# '-;-' means no fusion; only 'Yes' means a real fused construct
pos_fus = df_pos_p["Fusion"].str.contains("yes", case=False)
neg_fus = df_neg_p["Fusion"].str.contains("yes", case=False)

print("Positive fusion rows (would be dropped):", pos_fus.sum())   # expect 59
print(df_pos_p.loc[pos_fus, "Protein name"].value_counts().head(10), "\n")
print("Negative fusion rows (would be dropped):", neg_fus.sum())   # expect 10
print(df_neg_p.loc[neg_fus, "Protein name"].value_counts().head(10))

Positive fusion rows (would be dropped): 59
Protein name
FUS_m + Hsp27-αBc_m                        6
SUMO-LAF_m + streptavidin                  5
SUMO-FUS_m + SUMO-LAF_m + streptavidin     4
FUS_m + αBc-Hsp27_m                        4
SUMO-FUS_m + rEGFP-FUS_m + streptavidin    3
SUMO-FUS_m + rEGFP-LAF_m + streptavidin    3
SUMO-LAF_m + rEGFP-LAF_m + streptavidin    3
SUMO-LAF_m + SUMO-LAF_m                    3
SUMO-LAF_m + rEGFP-FUS_m + streptavidin    2
SUMO-TAF_m + SUMO-LAF_m + streptavidin     2
Name: count, dtype: int64 

Negative fusion rows (would be dropped): 10
Protein name
FUS_m + Hsp27-αBc_m         3
FUS-TPX2_m + tubulin        1
XMAP215-TPX2_m + tubulin    1
BuGZ-TPX2_m + tubulin       1
FUS-TPX2_m + BSA            1
XMAP215-TPX2_m + BSA        1
BuGZ-TPX2_m + BSA           1
FUS_m + αBc-Hsp27_m         1
Name: count, dtype: int64


In [32]:
df_pos_p = df_pos_p[~pos_fus].copy()
df_neg_p = df_neg_p[~neg_fus].copy()

print("New Positive shape:", df_pos_p.shape)   # expect 624 rows
print("New Negative shape:", df_neg_p.shape)   # expect 171 rows

# Fusion is all '-' now, so drop it
df_pos_p = df_pos_p.drop(columns=["Fusion"])
df_neg_p = df_neg_p.drop(columns=["Fusion"])
print(list(df_pos_p.columns))

New Positive shape: (624, 9)
New Negative shape: (171, 9)
['PSID', 'Entry ID', 'Components type', 'Protein ID', 'Protein name', 'Phase separation', 'PMID', 'with_nucleic_acid']


In [33]:
prot_pos_path = os.path.join(base_dir, "phase_separation_Unambiguous", "protein.xls")
prot_neg_path = os.path.join(base_dir, "No_phase_separation_Unambiguous", "protein.xls")

prot_table = pd.concat([pd.read_excel(prot_pos_path), pd.read_excel(prot_neg_path)], ignore_index=True)
prot_table.columns = prot_table.columns.str.strip()
prot_table = prot_table.drop_duplicates("PID").set_index("PID")   # PID = the p0001-style ID

print("Proteins in combined table:", len(prot_table))   # expect 541
print(prot_table[["Protein name abbreviation", "Gene name", "Uniprot ID", "Species"]].head())

Proteins in combined table: 541
      Protein name abbreviation                Gene name Uniprot ID  \
PID                                                                   
p0001                     LAF-1         laf-1;Y71H2AM.19     D0PV95   
p0002                       FUS                  FUS;TLS     P35637   
p0003                      Whi3             AGOS_ABL158C     Q75E28   
p0004                      Pub1  PUB1;RNP1;YNL016W;N2842     P32588   
p0005                      Lsm4  LSM4;SDB23;USS1;YER112W     P40070   

                        Species  
PID                              
p0001    Caenorhabditis elegans  
p0002              Homo sapiens  
p0003           Ashbya gossypii  
p0004  Saccharomyces cerevisiae  
p0005  Saccharomyces cerevisiae  


In [34]:
def split_ids(val):
    return [p.strip() for p in re.split(r"[;,]", str(val)) if p.strip()]

df_pos_p["pid_list"] = df_pos_p["Protein ID"].apply(split_ids)
df_neg_p["pid_list"] = df_neg_p["Protein ID"].apply(split_ids)

for name, df in [("POSITIVE", df_pos_p), ("NEGATIVE", df_neg_p)]:
    all_ids = {p for L in df["pid_list"] for p in L}
    missing = all_ids - set(prot_table.index)
    print(f"{name}: {len(all_ids)} unique protein IDs, missing from table: {len(missing)}")   # expect 0 missing

POSITIVE: 221 unique protein IDs, missing from table: 0
NEGATIVE: 103 unique protein IDs, missing from table: 0


In [35]:
# keep a row only if EVERY protein in it is human
def all_human(pids):
    return all(prot_table.loc[p, "Species"] == "Homo sapiens" for p in pids)

pos_h_mask = df_pos_p["pid_list"].apply(all_human)
neg_h_mask = df_neg_p["pid_list"].apply(all_human)

print("Positive rows with a non-human protein (would be dropped):", (~pos_h_mask).sum())
print("Negative rows with a non-human protein (would be dropped):", (~neg_h_mask).sum())
print("Positive all-human rows:", pos_h_mask.sum())   # expect 309
print("Negative all-human rows:", neg_h_mask.sum())   # expect 78

Positive rows with a non-human protein (would be dropped): 315
Negative rows with a non-human protein (would be dropped): 93
Positive all-human rows: 309
Negative all-human rows: 78


In [36]:
# keep a row only if EVERY protein in it is human
df_pos_h = df_pos_p[pos_h_mask].copy()
df_neg_h = df_neg_p[neg_h_mask].copy()

print("New Positive shape:", df_pos_h.shape)   # expect 309 rows
print("New Negative shape:", df_neg_h.shape)   # expect 78 rows

# verify only human proteins are left
species_left = {prot_table.loc[p, "Species"] for df in [df_pos_h, df_neg_h] for L in df["pid_list"] for p in L}
print("Species left:", species_left)   # should be only {'Homo sapiens'}

New Positive shape: (309, 9)
New Negative shape: (78, 9)
Species left: {'Homo sapiens'}


In [37]:
def to_uniprot(pids):
    # strip isoform endings (P09651-2 -> P09651) so IDs match other databases
    return ";".join(str(prot_table.loc[p, "Uniprot ID"]).strip().split("-")[0] for p in pids)

df_pos_h["UniProt ID"] = df_pos_h["pid_list"].apply(to_uniprot)
df_neg_h["UniProt ID"] = df_neg_h["pid_list"].apply(to_uniprot)

# check: no missing IDs, and same number of UniProt IDs as protein IDs in every row
for name, df in [("POSITIVE", df_pos_h), ("NEGATIVE", df_neg_h)]:
    missing = df["UniProt ID"].str.contains("nan|-", regex=True).sum()
    counts_match = (df["UniProt ID"].str.split(";").apply(len) == df["pid_list"].apply(len)).all()
    print(f"{name}: rows with missing UniProt: {missing} | counts match: {counts_match}")   # expect 0 | True

df_pos_h[["PSID", "Protein ID", "Protein name", "UniProt ID"]].head(10)

POSITIVE: rows with missing UniProt: 0 | counts match: True
NEGATIVE: rows with missing UniProt: 0 | counts match: True


,PSID,Protein ID,Protein name,UniProt ID
37,PS00000065,p0008;p0002,hnRNPA1 + FUS_m,P09651;P35637
76,PS00000119,p0002;p0108,FUS_m + hRPB1_m,P35637;P24928
83,PS00000127,p0002,FUS_m + FUS_m,P35637
94,PS00000144,p0150;p0002,TAF15_m + FUS_m,Q92804;P35637
118,PS00000177,p0002;p0148,FUS + Imp5,P35637;O00410
119,PS00000178,p0002;p0148,FUS + Imp5,P35637;O00410
120,PS00000179,p0002;p0147,FUS + PRMT1,P35637;Q99873
121,PS00000180,p0002;p0147,FUS + PRMT1,P35637;Q99873
122,PS00000184,p0002;p0151,FUS + TNPO1,P35637;Q92973
135,PS00000205,p0002;p0149,FUS_m + EWSR1,P35637;Q01844


In [38]:
pos_n = df_pos_h["pid_list"].apply(lambda L: len(set(L)))
neg_n = df_neg_h["pid_list"].apply(lambda L: len(set(L)))

print("Positive rows with only 1 distinct protein (would be dropped):", (pos_n < 2).sum())   # expect 51
print(df_pos_h.loc[pos_n < 2, "Protein name"].value_counts().head(8), "\n")
print("Negative rows with only 1 distinct protein (would be dropped):", (neg_n < 2).sum())   # expect 2

Positive rows with only 1 distinct protein (would be dropped): 51
Protein name
FUS_m + FUS_m            22
Tau_m + Tau_m             9
Tau + Tau_m               7
hnRNPDL_m + hnRNPDL_m     4
MeCP2_m + MeCP2           3
UBQLN2_m + UBQLN2_m       2
EWSR1_m + EWSR1_m         1
TAF15_m + TAF15_m         1
Name: count, dtype: int64 

Negative rows with only 1 distinct protein (would be dropped): 2


In [39]:
df_pos_h = df_pos_h[pos_n >= 2].copy()
df_neg_h = df_neg_h[neg_n >= 2].copy()

print("New Positive shape:", df_pos_h.shape)   # expect 258 rows
print("New Negative shape:", df_neg_h.shape)   # expect 76 rows

New Positive shape: (258, 10)
New Negative shape: (76, 10)


In [40]:
df_pos_h["phase_separated"] = 1
df_neg_h["phase_separated"] = 0

llps_clean = pd.concat([df_pos_h, df_neg_h], ignore_index=True)

final_cols = ["PSID", "phase_separated", "Protein ID", "Protein name", "UniProt ID", "PMID"]
llps_clean = llps_clean[final_cols]
llps_clean["PMID"] = llps_clean["PMID"].astype(str).str.replace(".0", "", regex=False)

out_file = "llps_human_protein_clean.csv"
llps_clean.to_csv(out_file, index=False)

print(f"Saved {out_file}: {len(llps_clean)} experiments")   # expect 334
print("Phase separated:", (llps_clean.phase_separated == 1).sum(), "| Not:", (llps_clean.phase_separated == 0).sum())   # expect 258 | 76
print("Unique proteins:", len({u for s in llps_clean["UniProt ID"] for u in s.split(";")}))
print("Papers:", llps_clean["PMID"].nunique())
llps_clean.head()

Saved llps_human_protein_clean.csv: 334 experiments
Phase separated: 258 | Not: 76
Unique proteins: 93
Papers: 48


,PSID,phase_separated,Protein ID,Protein name,UniProt ID,PMID
0,PS00000065,1,p0008;p0002,hnRNPA1 + FUS_m,P09651;P35637,26412307
1,PS00000119,1,p0002;p0108,FUS_m + hRPB1_m,P35637;P24928,26455390
2,PS00000144,1,p0150;p0002,TAF15_m + FUS_m,Q92804;P35637,29961577
3,PS00000177,1,p0002;p0148,FUS + Imp5,P35637;O00410,29677514
4,PS00000178,1,p0002;p0148,FUS + Imp5,P35637;O00410,29677514
